# EML 2.2.0 → JSON-LD Converter — LifeWatch ERIC NEW spec v5.2
**First use or after update:** Runtime → Restart runtime → Runtime → Run all

### Identifier & sameAs strategy (confirmed from real EML):
| EML source | JSON-LD output | Logic |
|-----------|----------------|-------|
| `@packageId` UUID | `@id` + `url` | Always the canonical record UUID |
| `alternateIdentifier` URLs (non-UUID) | `sameAs` | Handle, DOI, and other access URLs |
| `distribution/online/url` | `sameAs` (merged) | Additional download links |


In [ ]:
import subprocess, sys
for pkg in ["lxml","tabulate"]:
    try: __import__(pkg)
    except ImportError: subprocess.check_call([sys.executable,"-m","pip","install","-q",pkg])
print("EML2JsonLD v5 ready")


In [ ]:
import json, re, os
from copy import deepcopy
from typing import Optional
from lxml import etree
from tabulate import tabulate


In [ ]:
JSONLD_CONTEXT = {"@vocab": "https://schema.org/"}

_CC_RE = re.compile(
    r"https?://creativecommons\.org/licenses/[a-z\-]+/[\d\.]+/?",
    re.IGNORECASE,
)

UUID_RE = re.compile(
    r"[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}"
)

# Fixed host for @id and url — always the LifeWatch metadata catalogue
CATALOGUE_HOST = "https://metadatacatalogue.lifewatch.eu"

def _host(url):
    m = re.match(r"(https?://[^/]+)", url)
    return m.group(1) if m else CATALOGUE_HOST

def _is_pure_uuid(s):
    s = s.strip()
    return bool(UUID_RE.fullmatch(s))


In [ ]:
def _local(el):
    if callable(el.tag): return ""
    return etree.QName(el).localname

def _text(el):
    if el is None: return None
    t = (el.text or "").strip()
    return t if t else None

def direct_children(parent, tag):
    return [c for c in parent if _local(c) == tag]

def first_child(parent, tag):
    for c in parent:
        if _local(c) == tag: return c
    return None

def find_first(root, tag):
    for el in root.iter():
        if _local(el) == tag: return el
    return None

def find_all(root, tag):
    return [el for el in root.iter() if _local(el) == tag]

def first_text(root, tag):
    return _text(find_first(root, tag))

def all_texts(root, tag):
    return [t for el in root.iter()
            if _local(el) == tag for t in [_text(el)] if t]

def para_text(el):
    if el is None: return None
    paras = [_text(p) for p in el if _local(p) == "para"]
    paras = [p for p in paras if p]
    if paras: return " ".join(paras)
    text = " ".join(el.itertext()).strip()
    return text if text else None

def build_id_map(root):
    id_map = {}
    for el in root.iter():
        eid = el.get("id")
        if eid: id_map[eid] = el
    return id_map

def resolve(el, id_map):
    ref_el = first_child(el, "references")
    if ref_el is not None and ref_el.text:
        target = id_map.get(ref_el.text.strip())
        if target is not None: return deepcopy(target)
    return el


In [ ]:
def parse_party(el, id_map=None):
    if id_map: el = resolve(el, id_map)
    given  = first_text(el, "givenName")
    family = first_text(el, "surName")
    org    = first_text(el, "organizationName")
    pos    = first_text(el, "positionName")
    node   = {}
    if given or family:
        node["@type"] = "Person"
        node["name"]  = " ".join(p for p in [given, family] if p)
        if given:  node["givenName"]  = given
        if family: node["familyName"] = family
        if org:    node["affiliation"] = {"@type": "Organization", "name": org}
        if pos:    node["jobTitle"]    = pos
    elif org:
        node["@type"] = "Organization"
        node["name"]  = org
        if pos: node["jobTitle"] = pos
    elif pos:
        node["@type"] = "Person"
        node["name"]  = pos
    else:
        return {}
    email = first_text(el, "electronicMailAddress")
    if email: node["email"] = email
    url = first_text(el, "onlineUrl")
    if url: node["url"] = url
    for uid_el in direct_children(el, "userId"):
        uid_val = _text(uid_el)
        if not uid_val: continue
        uid_dir  = (uid_el.get("directory") or "").lower()
        is_orcid = "orcid" in uid_dir or "orcid.org" in uid_val.lower()
        if is_orcid:
            uri = uid_val if uid_val.startswith("http") else "https://orcid.org/" + uid_val
            node["identifier"] = {"@type":"PropertyValue","propertyID":"ORCID","value":uri}
        else:
            node["sameAs"] = uid_val
        break
    addr_el = find_first(el, "address")
    if addr_el is not None:
        addr = {"@type": "PostalAddress"}
        sv = first_text(addr_el, "deliveryPoint")
        ct = first_text(addr_el, "city")
        ar = first_text(addr_el, "administrativeArea")
        pc = first_text(addr_el, "postalCode")
        co = first_text(addr_el, "country")
        if sv: addr["streetAddress"]   = sv
        if ct: addr["addressLocality"] = ct
        if ar: addr["addressRegion"]   = ar
        if pc: addr["postalCode"]      = pc
        if co: addr["addressCountry"]  = co
        if len(addr) > 1: node["address"] = addr
    return node


In [ ]:
def parse_geographic(geo_el):
    node = {"@type": "Place"}
    desc_el = first_child(geo_el, "geographicDescription")
    if desc_el is None:
        desc_el = find_first(geo_el, "geographicDescription")
    desc = para_text(desc_el) or _text(desc_el)
    if desc: node["name"] = desc
    bc = first_child(geo_el, "boundingCoordinates")
    if bc is None:
        bc = find_first(geo_el, "boundingCoordinates")
    if bc is not None:
        w = first_text(bc, "westBoundingCoordinate")
        e = first_text(bc, "eastBoundingCoordinate")
        n = first_text(bc, "northBoundingCoordinate")
        s = first_text(bc, "southBoundingCoordinate")
        if all([w, e, n, s]):
            node["geo"] = {"@type": "GeoShape", "box": s+" "+w+" "+n+" "+e}
    return node

def parse_temporal(temp_el):
    rod = find_first(temp_el, "rangeOfDates")
    if rod is not None:
        dates = all_texts(rod, "calendarDate")
        if len(dates) >= 2: return dates[0] + "/" + dates[1]
        if len(dates) == 1: return dates[0]
    sdt = find_first(temp_el, "singleDateTime")
    if sdt is not None:
        return first_text(sdt, "calendarDate") or first_text(sdt, "time")
    return None

def parse_attribute(attr_el):
    node = {"@type": "PropertyValue"}
    for tag in ("semanticAnnotation", "annotation"):
        for ann_el in direct_children(attr_el, tag):
            val_el = find_first(ann_el, "valueURI")
            if val_el is not None:
                uri = _text(val_el)
                if uri and uri.startswith("http"):
                    node["@id"] = uri
                    break
        if "@id" in node: break
    name = first_text(attr_el, "attributeName")
    if name: node["name"] = name
    defn = first_text(attr_el, "attributeDefinition")
    if defn: node["description"] = defn
    std  = (first_text(attr_el, "standardUnit")
            or first_text(attr_el, "attributeStandardUnit"))
    cus  = first_text(attr_el, "customUnit")
    if std: node["unitCode"] = std
    if cus: node["unitText"] = cus
    fmt = first_text(attr_el, "formatString")
    if fmt: node["encodingFormat"] = fmt
    mn = first_text(attr_el, "minimum")
    mx = first_text(attr_el, "maximum")
    if mn: node["minValue"] = mn
    if mx: node["maxValue"] = mx
    return node


In [ ]:
class EML2JsonLD:
    """
    EML 2.2.0 -> JSON-LD  (LifeWatch ERIC NEW spec v5.2)

    UUID / @id / url strategy
    -------------------------
    The canonical record UUID ALWAYS comes from @packageId attribute on <eml>.
    <alternateIdentifier> contains OTHER identifiers (handle, DOI) -> sameAs.

    sameAs strategy
    ---------------
    Sources (all merged, deduplicated):
      1. <alternateIdentifier> values that are URLs (http/https) or look like
         external identifiers (DOI, handle) — NOT pure UUIDs
      2. All <url> elements inside any <distribution> anywhere in the tree
    """

    def __init__(self, xml_source):
        if isinstance(xml_source, str) and os.path.exists(xml_source):
            with open(xml_source, 'rb') as fh: raw = fh.read()
        elif isinstance(xml_source, bytes): raw = xml_source
        else: raise TypeError('xml_source must be file path or XML bytes')
        parser = etree.XMLParser(remove_comments=True, recover=True)
        self._root   = etree.fromstring(raw, parser=parser)
        self._id_map = build_id_map(self._root)
        self._log    = []
        self._result = None

    def _ds(self):
        ds = find_first(self._root, 'dataset')
        return ds if ds is not None else self._root

    def _ok(self, field, target):
        self._log.append({'field': field, 'status': 'OK', 'target': target})

    def _partial(self, field, reason):
        self._log.append({'field': field, 'status': 'PARTIAL', 'target': reason})

    def _lost(self, field, reason):
        self._log.append({'field': field, 'status': 'LOST', 'target': reason})

    def _map_identifiers(self, doc):
        """
        UUID source priority (confirmed by manager):
          1. UUID extracted from the DOI <alternateIdentifier>
             (doi.org/10.48372/{UUID}) — this is the GeoNetwork catalogue UUID
          2. Fallback: UUID from @packageId (when DOI has no UUID, e.g. HVNX-R248)

          @id = https://metadatacatalogue.lifewatch.eu/srv/api/records/{UUID}
          url = https://metadatacatalogue.lifewatch.eu/srv/eng/catalog.search#/metadata/{UUID}
          Host always = LifeWatch ERIC metadata catalogue (CATALOGUE_HOST constant).

          All <alternateIdentifier> URL values go to sameAs (handle + DOI)
          — handled in _map_sameas().
        """
        ds  = self._ds()
        pkg = self._root.get('packageId') or self._root.get('id') or ''

        # Priority 1: UUID from DOI alternateIdentifier (doi.org/10.48372/{UUID})
        uuid = None
        uuid_src = ''
        for el in direct_children(ds, 'alternateIdentifier'):
            val = (el.text or '').strip()
            if 'doi.org' in val:
                m = UUID_RE.search(val)
                if m:
                    uuid = m.group(0)
                    uuid_src = 'DOI alternateIdentifier: ' + val
                    break

        # Priority 2: UUID from packageId (fallback when DOI has no UUID)
        if not uuid:
            m2 = UUID_RE.search(pkg)
            if m2:
                uuid = m2.group(0)
                uuid_src = 'packageId (DOI had no UUID): ' + pkg

        if uuid:
            doc['@id'] = CATALOGUE_HOST + '/srv/api/records/' + uuid
            doc['url'] = CATALOGUE_HOST + '/srv/eng/catalog.search#/metadata/' + uuid
            self._ok('alternateIdentifier/packageId -> UUID',
                     '@id + url UUID ' + uuid + ' from ' + uuid_src)
        elif pkg.startswith('http'):
            doc['@id'] = pkg
            self._ok('/eml/@packageId', '@id = raw packageId (no UUID anywhere)')
            self._lost('url', 'No UUID found - url not set')
        else:
            self._lost('/eml/@packageId',
                       'No UUID in DOI alternateIdentifier or packageId')

    def _map_sameas(self, doc):
        """
        Per NEW sheet row 18:
          sameAs = plain URL string or array of plain URL strings.

        Sources (merged, deduplicated):
          1. <distribution><online><url> anywhere in the dataset tree
             (excluding distribution inside methods/software/project contexts)
          2. DOI <alternateIdentifier> values (doi.org URLs)
             — the handle URL is skipped when it already appears via distribution

        Fallback: if no distribution exists, use ALL non-UUID alternateIdentifiers
        (handle + DOI) so sameAs is never empty when access URLs exist.

        Result: plain string (1 URL) or plain array (multiple URLs).
        No {"@type":"URL","@id":...} wrappers — plain strings only.
        """
        ds   = self._ds()
        urls = []

        def _add(u):
            u = (u or '').strip()
            if u and u not in urls:
                urls.append(u)

        _EXCLUDED_ANCESTORS = {'methods', 'methodStep', 'software',
                                'implementation', 'project', 'literature'}

        # Source 1: distribution/online/url (primary — per spec row 18)
        dist_found = False
        for dist_el in ds.iter():
            if _local(dist_el) != 'distribution':
                continue
            ancestor_tags = {_local(a) for a in dist_el.iterancestors()}
            if ancestor_tags & _EXCLUDED_ANCESTORS:
                continue
            for child in dist_el:
                ctag = _local(child)
                if ctag == 'online':
                    for url_el in child:
                        if _local(url_el) == 'url':
                            u = _text(url_el)
                            if u:
                                _add(u)
                                dist_found = True
                elif ctag == 'url':
                    u = _text(child)
                    if u:
                        _add(u)
                        dist_found = True

        # Source 2: all non-UUID alternateIdentifiers
        # When distribution found: add DOI only (handle already via distribution)
        # When no distribution: add ALL URL alternateIdentifiers in original order
        for el in direct_children(ds, 'alternateIdentifier'):
            val = (el.text or '').strip()
            if not val or _is_pure_uuid(val):
                continue
            if dist_found:
                # Only add DOI — handle already covered by distribution URL
                if 'doi.org' in val:
                    _add(val)
            else:
                # No distribution — add all URL alternateIdentifiers
                _add(val)

        if not urls:
            self._lost('distribution/online/url',
                       'No distribution URL or alternateIdentifier URL found')
            return

        doc['sameAs'] = urls[0] if len(urls) == 1 else urls
        self._ok('distribution/online/url',
                 'sameAs (plain URL string or array) = ' + str(doc['sameAs']))

    def _map_basic(self, doc):
        ds = self._ds()
        titles = [el.text.strip() for el in direct_children(ds, 'title')
                  if (el.text or '').strip()]
        if titles:
            doc['name'] = titles[0]
            self._ok('title', 'name')
        else:
            self._lost('title', 'No <title>')
        short = first_text(ds, 'shortName')
        if short:
            doc['alternateName'] = short
            self._ok('shortName', 'alternateName')
        ab_el = find_first(ds, 'abstract')
        if ab_el is not None:
            ab = para_text(ab_el)
            if ab:
                doc['description'] = ab
                self._ok('abstract/para', 'description')
            else:
                self._lost('abstract/para', 'Empty <abstract>')
        else:
            self._lost('abstract/para', 'No <abstract>')
        pd = first_text(ds, 'pubDate')
        if pd:
            doc['datePublished'] = pd
            self._ok('pubDate', 'datePublished')
        else:
            self._lost('pubDate', 'No <pubDate>')
        lang = first_text(ds, 'language')
        if lang:
            doc['inLanguage'] = lang
            self._ok('language', 'inLanguage')
        purp = para_text(find_first(ds, 'purpose'))
        if purp:
            doc['purpose'] = purp
            self._ok('purpose', 'purpose')
        ai = para_text(find_first(ds, 'additionalInfo'))
        if ai:
            doc['additionalInfo'] = ai
            self._ok('additionalInfo', 'additionalInfo')
        maint_el = find_first(ds, 'maintenance')
        if maint_el is not None:
            maint = {}
            t = para_text(find_first(maint_el, 'description'))
            if t: maint['description'] = t
            freq = first_text(maint_el, 'maintenanceUpdateFrequency')
            if freq: maint['frequency'] = freq
            if maint:
                doc['maintenance'] = maint
                self._ok('maintenance', 'maintenance')

    def _map_parties(self, doc):
        ds   = self._ds()
        imap = self._id_map
        creators = direct_children(ds, 'creator')
        if creators:
            nodes = [n for n in (parse_party(c, imap) for c in creators) if n]
            doc['creator'] = nodes if len(nodes) > 1 else nodes[0]
            self._ok('creator', 'creator (Person/Organization)')
        else:
            self._lost('creator', 'No <creator>')
        contacts = direct_children(ds, 'contact')
        if contacts:
            self._partial('contact', 'contactPoint invalid on Dataset per validator')
        assoc = direct_children(ds, 'associatedParty')
        if assoc:
            nodes = [n for n in (parse_party(a, imap) for a in assoc) if n]
            doc['contributor'] = nodes if len(nodes) > 1 else nodes[0]
            self._ok('associatedParty', 'contributor')
        pub_el = find_first(ds, 'publisher')
        if pub_el is not None:
            node = parse_party(pub_el, imap)
            if node:
                doc['publisher'] = node
                self._ok('publisher', 'publisher')
        # metadataProvider has no schema.org equivalent — dropped per LifeWatch ERIC spec
        # provider is always the hardcoded LifeWatch ERIC organisation block
        doc['provider'] = {
            "@type": "Organization",
            "name": "e-Science and Technology European Infrastructure for Biodiversity and Ecosystem Research",
            "alternateName": "LifeWatch ERIC",
            "identifier": "https://ror.org/04c04g438",
            "url": "https://www.lifewatch.eu",
            "email": "communications@lifewatch.eu"
        }
        self._ok('provider', 'provider (hardcoded LifeWatch ERIC organisation block)')

    def _map_rights(self, doc):
        ds = self._ds()
        for lic_el in direct_children(ds, 'licensed'):
            lic_url = first_text(lic_el, 'url')
            if lic_url:
                doc['license'] = lic_url.rstrip('/').replace('http://', 'https://') + '/'
                self._ok('CC URL in rights text', 'license (from <licensed><url>)')
                return
        ir_el = find_first(ds, 'intellectualRights')
        if ir_el is not None:
            t = para_text(ir_el)
            if t:
                m = _CC_RE.search(t)
                if m:
                    doc['license'] = m.group(0).rstrip('/').replace('http://', 'https://') + '/'
                    self._ok('CC URL in rights text', 'license (CC URL from text)')
                else:
                    self._partial('CC URL in rights text', 'No CC URL found')
        else:
            self._lost('CC URL in rights text', 'No <intellectualRights> or <licensed>')

    def _map_keywords(self, doc):
        ds  = self._ds()
        out = []
        for ks in find_all(ds, 'keywordSet'):
            thes = first_text(ks, 'keywordThesaurus')
            for kw_el in direct_children(ks, 'keyword'):
                kw = _text(kw_el)
                if not kw: continue
                if thes:
                    out.append({'@type': 'DefinedTerm', 'name': kw,
                                'inDefinedTermSet': {'@type': 'DefinedTermSet', 'name': thes}})
                else:
                    out.append(kw)
        if out:
            doc['keywords'] = out
            self._ok('keywordSet/keyword + keywordThesaurus', 'keywords[]')
        else:
            self._lost('keywordSet/keyword', 'No <keywordSet>')

    def _map_coverage(self, doc):
        ds     = self._ds()
        cov_el = find_first(ds, 'coverage')
        if cov_el is None:
            self._lost('coverage', 'No <coverage>')
            return
        geo_els = find_all(cov_el, 'geographicCoverage')
        if geo_els:
            places = [parse_geographic(g) for g in geo_els]
            doc['spatialCoverage'] = places if len(places) > 1 else places[0]
            self._ok('geographicCoverage', 'spatialCoverage (Place + GeoShape box)')
        else:
            self._lost('geographicCoverage', 'Not present')
        temp_el = find_first(cov_el, 'temporalCoverage')
        if temp_el is not None:
            interval = parse_temporal(temp_el)
            if interval:
                doc['temporalCoverage'] = interval
                self._ok('temporalCoverage', 'temporalCoverage (ISO 8601 interval)')
            else:
                self._lost('temporalCoverage', 'No dates parsed')
        else:
            self._lost('temporalCoverage', 'Not present')

    def _map_entities(self, doc):
        ds        = self._ds()
        all_attrs = []
        for dt_el in direct_children(ds, 'dataTable'):
            for a in find_all(dt_el, 'attribute'):
                all_attrs.append(parse_attribute(a))
        for oe_el in direct_children(ds, 'otherEntity'):
            for a in find_all(oe_el, 'attribute'):
                all_attrs.append(parse_attribute(a))
        seen   = set()
        unique = []
        for a in all_attrs:
            key = a.get('@id') or a.get('name', '')
            if key and key not in seen:
                seen.add(key)
                unique.append(a)
        if unique:
            doc['variableMeasured'] = unique
            self._ok('dataTable/attribute', 'variableMeasured[] (all tables merged, deduped)')
        else:
            self._lost('dataTable/attribute', 'No <attribute> elements found')

    def _map_project(self, doc):
        ds      = self._ds()
        proj_el = find_first(ds, 'project')
        if proj_el is None: return
        fund_el = find_first(proj_el, 'funding')
        if fund_el is not None:
            t = para_text(fund_el)
            if t:
                doc['funder'] = {'@type': 'Organization', 'name': t}
                self._ok('project/funding', 'funder (Organization)')
        pt = first_text(proj_el, 'title')
        if pt:
            self._partial('project/title', 'No schema.org Dataset slot for project title')

    def convert(self):
        self._log = []
        doc = {'@context': JSONLD_CONTEXT, '@type': 'Dataset'}
        self._map_identifiers(doc)   # @id, url  <- packageId UUID
        self._map_sameas(doc)        # sameAs    <- alternateIdentifier URLs + distribution URLs
        self._map_basic(doc)
        self._map_parties(doc)
        self._map_rights(doc)
        self._map_keywords(doc)
        self._map_coverage(doc)
        self._map_entities(doc)
        self._map_project(doc)
        self._result = doc
        return doc

    def save(self, path):
        if self._result is None: self.convert()
        with open(path, 'w', encoding='utf-8') as fh:
            json.dump(self._result, fh, indent=2, ensure_ascii=False)
        print('Saved -> ' + path)

    def print_loss_report(self):
        if not self._log: self.convert()
        mapped  = sum(1 for r in self._log if r['status'] == 'OK')
        partial = sum(1 for r in self._log if r['status'] == 'PARTIAL')
        lost    = sum(1 for r in self._log if r['status'] == 'LOST')
        total   = len(self._log)
        pct     = round(100 * mapped / total) if total else 0
        sep = '=' * 80
        print(sep)
        print('MAPPING REPORT  EML 2.2.0 -> JSON-LD  (LifeWatch ERIC NEW spec v5.2)')
        print(sep)
        print('Fields evaluated : ' + str(total))
        print('Fully mapped     : ' + str(mapped) + '  (' + str(pct) + '%)')
        print('Partial          : ' + str(partial))
        print('Lost             : ' + str(lost))
        print(sep)
        rows = [[r['field'], r['status'], r['target']] for r in self._log]
        print(tabulate(rows,
                       headers=['EML Field', 'Status', 'Mapped to / Reason'],
                       tablefmt='simple'))
        print(sep)


print('EML2JsonLD v5.2 loaded')
print('  @id/url  <- packageId UUID')
print('  sameAs   <- alternateIdentifier URLs + distribution URLs (merged)')
print('  provider <- hardcoded LifeWatch ERIC block (metadataProvider dropped)')


In [ ]:
def validate_jsonld(doc):
    warnings = []
    if doc.get("@context") != {"@vocab": "https://schema.org/"}:
        warnings.append('@context should be {"@vocab": "https://schema.org/"}')
    if doc.get("@type") != "Dataset":
        warnings.append("@type should be Dataset")
    if not doc.get("@id"):
        warnings.append("Missing @id")
    elif "/srv/api/records/" not in str(doc.get("@id", "")):
        warnings.append("@id should contain /srv/api/records/{uuid}, got: " + str(doc.get("@id")))
    if not doc.get("url"):
        warnings.append("Missing url (catalog search page)")
    if not doc.get("name"):
        warnings.append("Missing name")
    if not doc.get("description"):
        warnings.append("Missing description")
    if not doc.get("creator"):
        warnings.append("Missing creator")
    same_as = doc.get("sameAs")
    if same_as is not None:
        items = same_as if isinstance(same_as, list) else [same_as]
        for i, item in enumerate(items):
            if isinstance(item, dict):
                warnings.append("sameAs[" + str(i) + "] must be plain URL string not object")
    for key in ("about", "identifier", "contactPoint",
                "dcterms:title", "schema:name", "distribution"):
        if key in doc:
            warnings.append("Forbidden key found: " + key)
    return warnings


In [ ]:
def run(xml_path=None, output_path="output.jsonld"):
    if xml_path is None:
        try:
            from google.colab import files
            print("Upload your EML 2.2.0 XML file ...")
            uploaded = files.upload()
            if not uploaded:
                print("No file uploaded.")
                return None
            fname     = list(uploaded.keys())[0]
            xml_bytes = uploaded[fname]
            print("Uploaded: " + fname)
        except Exception:
            print("Not in Colab — use: result = run(xml_path='/content/your_file.xml')")
            return None
    else:
        fname = os.path.basename(xml_path)
        with open(xml_path, "rb") as fh:
            xml_bytes = fh.read()
        print("Loaded: " + fname)

    print("Converting ...")
    conv   = EML2JsonLD(xml_bytes)
    result = conv.convert()
    conv.save(output_path)

    issues = validate_jsonld(result)
    if issues:
        print("Validation warnings:")
        for w in issues: print("  - " + w)
    else:
        print("Validation passed — verify at https://validator.schema.org/")

    try:
        from IPython.display import display, JSON
        print("JSON-LD preview:")
        display(JSON(result))
    except Exception:
        print(json.dumps(result, indent=2, ensure_ascii=False)[:3000])

    conv.print_loss_report()

    try:
        from google.colab import files
        files.download(output_path)
        print("Downloaded: " + output_path)
    except Exception:
        print("Output saved to: " + output_path)

    return result


## Run
Click **Runtime → Run all**, upload your EML XML when prompted.

Or: `result = run(xml_path='/content/your_file.xml')`


In [ ]:
result = run()
